**v2: load, feed, include ewma**

### Libraries: 

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from dtw import dtw
import copy

from keras.models import Sequential
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Dropout

import plotly.express as px

### Load Data: 

In [2]:
data = pd.read_pickle('../data/inlier_data.pkl')
data.head()

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index,load_Rotary_C5_temp,cr_numeric
0,2024-03-11 15:40:47.167578,2.0,18.886,10.1493,35.0,239.27,0,O0005(5303-005-C),N10,CR-2,0,0.014493,2
1,2024-03-11 15:40:56.190506,1.0,18.886,10.1493,35.0,239.27,0,O0005(5303-005-C),N10,CR-2,1,0.000000,2
2,2024-03-11 15:41:01.202792,1.0,16.250,3.9771,35.0,239.25,0,O0005(5303-005-C),N10,CR-2,2,0.000000,2
3,2024-03-11 15:42:17.411486,2.0,16.250,0.8980,35.0,10.78,0,O0005(5303-005-C),N10,CR-2,3,0.014493,2
4,2024-03-11 15:42:22.424109,1.0,16.250,0.8980,35.0,10.78,0,O0005(5303-005-C),N10,CR-2,4,0.000000,2


In [3]:
# data = data[['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]

data = data[['timestamp', 'load_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]

In [4]:
data = data.rename(columns={
    'load_Rotary_C5': 'load', 
    'pathfeedrate_Path_Path_1': 'feed', 
})

### Train-Test Split

In [5]:
cr_list = data['cr'].unique().tolist()

In [6]:
size = len(cr_list)

train_ratio = 0.8
train_size = int(size*train_ratio)
test_size = size - train_size

print('train_size: ', train_size)
print('test_size: ', test_size)

train_crs =  cr_list[0: train_size]
test_crs = cr_list[train_size: -1]

print('train_crs: ', train_crs)
print('test_crs: ', test_crs)

train_size:  9
test_size:  3
train_crs:  ['CR-2', 'CR-4', 'CR-5', 'CR-6', 'CR-7', 'CR-8', 'CR-9', 'CR-11', 'CR-12']
test_crs:  ['CR-13', 'CR-15']


## Feature Engineering: 

#### Adding ewma columns for load and feed: 

In [ ]:
ALPHA_SLOW = 0.05 
ALPHA_FAST = 0.1 

In [ ]:
data['load_ewma_slow'] = (
    data.groupby('cr')['load']
      .transform(
          lambda x: x.ewm(
              alpha=ALPHA_SLOW,
              adjust=False
          ).mean()
      )
)

In [ ]:
data['load_ewma_fast'] = (
    data.groupby('cr')['load']
      .transform(
          lambda x: x.ewm(
              alpha=ALPHA_FAST,
              adjust=False
          ).mean()
      )
)

In [ ]:
data['feed_ewma_slow'] = (
    data.groupby('cr')['feed']
      .transform(
          lambda x: x.ewm(
              alpha=ALPHA_SLOW,
              adjust=False
          ).mean()
      )
)

In [ ]:
data['feed_ewma_fast'] = (
    data.groupby('cr')['feed']
      .transform(
          lambda x: x.ewm(
              alpha=ALPHA_FAST,
              adjust=False
          ).mean()
      )
)

In [ ]:
# dfN10_load_filtered_cr.to_pickle("D:/baseline_improvement-main/params_reassessment_june2026/data/inlier_data.pkl")

In [ ]:
data.head() 

,timestamp,load,feed,execution,program_name,nsequence,cr,load_ewma_slow,load_ewma_fast,feed_ewma_slow,feed_ewma_fast
0,2024-03-11 15:40:47.167578,2.0,239.27,0,O0005(5303-005-C),N10,CR-2,2.000000,2.0000,239.270000,239.27000
1,2024-03-11 15:40:56.190506,1.0,239.27,0,O0005(5303-005-C),N10,CR-2,1.950000,1.9000,239.270000,239.27000
2,2024-03-11 15:41:01.202792,1.0,239.25,0,O0005(5303-005-C),N10,CR-2,1.902500,1.8100,239.269000,239.26800
3,2024-03-11 15:42:17.411486,2.0,10.78,0,O0005(5303-005-C),N10,CR-2,1.907375,1.8290,227.844550,216.41920
4,2024-03-11 15:42:22.424109,1.0,10.78,0,O0005(5303-005-C),N10,CR-2,1.862006,1.7461,216.991322,195.85528


#### Train-test split: 

In [ ]:
train_data = data[data['cr'].isin(train_crs)]
train_data.shape

(5379, 11)

In [ ]:
test_data = data[data['cr'].isin(test_crs)]
test_data.shape

(1267, 11)

In [18]:
training_cols = ['load', 'feed', 'load_ewma_slow', 'load_ewma_fast', 'feed_ewma_slow', 'feed_ewma_fast']

In [19]:
train_data = train_data[training_cols]
test_data = test_data[training_cols] 

In [20]:
train_data.head(10)

,load,feed,load_ewma_slow,load_ewma_fast,feed_ewma_slow,feed_ewma_fast
0,2.0,239.27,2.000000,2.000000,239.270000,239.270000
1,1.0,239.27,1.950000,1.900000,239.270000,239.270000
2,1.0,239.25,1.902500,1.810000,239.269000,239.268000
3,2.0,10.78,1.907375,1.829000,227.844550,216.419200
4,1.0,10.78,1.862006,1.746100,216.991322,195.855280
5,1.0,15.70,1.818906,1.671490,206.926756,177.839752
6,1.0,1.41,1.777961,1.604341,196.650919,160.196777
7,1.0,1.41,1.739063,1.543907,186.888873,144.318099
8,1.0,1.41,1.702109,1.489516,177.614929,130.027289
9,1.0,1.41,1.667004,1.440565,168.804683,117.165560


In [21]:
scaler2 = MinMaxScaler()
train_data_scaled = scaler2.fit_transform(train_data)
test_data_scaled = scaler2.transform(test_data)

In [22]:
type(train_data_scaled)

numpy.ndarray

In [ ]:
# temp_df = pd.DataFrame(train_data_scaled, columns=['load_Rotary_C5', 'pathfeedrate_Path_Path_1'])
# temp_df.head()

In [23]:
train_data_scaled[:10]

array([[0.01960784, 0.99281472, 0.0320105 , 0.03119055, 1.        ,
        1.        ],
       [0.        , 0.99281472, 0.03040998, 0.02807149, 1.        ,
        1.        ],
       [0.        , 0.99273165, 0.02888948, 0.02526434, 0.99999581,
        0.99999163],
       [0.01960784, 0.04381775, 0.02904553, 0.02585696, 0.95217399,
        0.90436118],
       [0.        , 0.04381775, 0.02759325, 0.02327127, 0.90674326,
        0.81829377],
       [0.        , 0.06425219, 0.02621359, 0.02094414, 0.8646138 ,
        0.7428923 ],
       [0.        , 0.00490094, 0.02490291, 0.01884973, 0.82159997,
        0.6690501 ],
       [0.        , 0.00490094, 0.02365777, 0.01696475, 0.78073684,
        0.60259212],
       [0.        , 0.00490094, 0.02247488, 0.01526828, 0.74191686,
        0.54277993],
       [0.        , 0.00490094, 0.02135113, 0.01374145, 0.70503788,
        0.48894897]])

In [24]:
# Used after adding features: 

def to_sequences(x, seq_size):
    x_values = []
    y_values = []
    for i in range(len(x) - seq_size):
        x_values.append(x[i : (i + seq_size)])
        y_values.append(x[i + seq_size])
    return np.array(x_values), np.array(y_values)

In [25]:
X_train, y_train = to_sequences(train_data_scaled, 10)
print(X_train.shape, y_train.shape)

X_test, y_test = to_sequences(test_data_scaled, 10)
print(X_test.shape, y_test.shape)

(5369, 10, 6) (5369, 6)
(1257, 10, 6) (1257, 6)


In [26]:
X_train[:1]

array([[[0.01960784, 0.99281472, 0.0320105 , 0.03119055, 1.        ,
         1.        ],
        [0.        , 0.99281472, 0.03040998, 0.02807149, 1.        ,
         1.        ],
        [0.        , 0.99273165, 0.02888948, 0.02526434, 0.99999581,
         0.99999163],
        [0.01960784, 0.04381775, 0.02904553, 0.02585696, 0.95217399,
         0.90436118],
        [0.        , 0.04381775, 0.02759325, 0.02327127, 0.90674326,
         0.81829377],
        [0.        , 0.06425219, 0.02621359, 0.02094414, 0.8646138 ,
         0.7428923 ],
        [0.        , 0.00490094, 0.02490291, 0.01884973, 0.82159997,
         0.6690501 ],
        [0.        , 0.00490094, 0.02365777, 0.01696475, 0.78073684,
         0.60259212],
        [0.        , 0.00490094, 0.02247488, 0.01526828, 0.74191686,
         0.54277993],
        [0.        , 0.00490094, 0.02135113, 0.01374145, 0.70503788,
         0.48894897]]])

## Model Training: 

In [27]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense, TimeDistributed
from tensorflow.keras.models import Model

class LSTMTimeSeriesAutoencoder:
    def __init__(self, sequence_length, num_dimensions, dim_weights):
        self.sequence_length = sequence_length
        self.num_dimensions = num_dimensions
        self.dim_weights = dim_weights
        self.model = self.build_model()

    def build_model(self):
        # Define input shape
        input_shape = (self.sequence_length, self.num_dimensions)

        # Encoder
        encoder_inputs = Input(shape=input_shape)
        encoder = LSTM(128, return_sequences=True)(encoder_inputs)
        encoder = LSTM(64, return_sequences=True)(encoder)
        encoder = LSTM(32, return_sequences=True)(encoder)
        encoder = LSTM(16, return_sequences=True)(encoder)
        encoder = LSTM(8, return_sequences=False)(encoder)

        # Repeat the latent vector for each time step
        decoder_inputs = RepeatVector(self.sequence_length)(encoder)

        # Decoder
        decoder = LSTM(8, return_sequences=True)(decoder_inputs)
        decoder = LSTM(16, return_sequences=True)(decoder)
        decoder = LSTM(32, return_sequences=True)(decoder)
        decoder = LSTM(64, return_sequences=True)(decoder)
        decoder = LSTM(128, return_sequences=True)(decoder)

        # Output layer
        output = TimeDistributed(Dense(self.num_dimensions))(decoder)

        # Define the model
        model = Model(inputs=encoder_inputs, outputs=output)

        return model

    def custom_loss(self, y_true, y_pred):
        # Define weights for each dimension
        weights = tf.constant(self.dim_weights, dtype=tf.float32)

        # Calculate mean squared error (MSE) for each dimension
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=0)

        # Multiply MSE by weights and sum across dimensions
        weighted_loss = tf.reduce_sum(mse * weights)

        return weighted_loss

    def compile_model(self):
        self.model.compile(optimizer='adam', loss=self.custom_loss)

    def summary(self):
        self.model.summary()

    def train(self, x_train, epochs, batch_size, callbacks):
        history = self.model.fit(x_train, x_train, epochs=epochs, batch_size=batch_size, verbose=1, validation_split=0.1,  callbacks=callbacks, shuffle=False)
        return history

    def predict(self, x):
        return self.model.predict(x)

    def save_weights(self, filepath):
        # Save model weights
        self.model.save_weights(filepath)

    def load_weights(self, filepath):
        # Load model weights
        self.model.load_weights(filepath)


In [34]:
from keras.callbacks import ModelCheckpoint

# Define the ModelCheckpoint callback
checkpoint = ModelCheckpoint(filepath='D:/baseline_improvement-main/params_reassessment_june2026/ml_artifacts/v3/baseline_v3.keras', 
                             monitor='val_loss', 
                             verbose=1, 
                             save_best_only=True, 
                             mode='min')

In [35]:
model = LSTMTimeSeriesAutoencoder(sequence_length= X_train.shape[1], num_dimensions=X_train.shape[2], dim_weights = [0.80, 0.05, 0.05, 0.05, 0.025, 0.025])

In [36]:
print(type(model))

<class '__main__.LSTMTimeSeriesAutoencoder'>


In [37]:
model.compile_model()
# type(model)
history = model.train(x_train=X_train, epochs=50, batch_size=64, callbacks=[checkpoint])

Epoch 1/50

76/76 [==============================] - ETA: 0s - loss: 0.5323
Epoch 1: val_loss improved from inf to 0.32105, saving model to D:/baseline_improvement-main/params_reassessment_june2026/ml_artifacts/v3\baseline_v3.keras
76/76 [==============================] - 19s 74ms/step - loss: 0.5323 - val_loss: 0.3211
Epoch 2/50
75/76 [============================>.] - ETA: 0s - loss: 0.2739
Epoch 2: val_loss improved from 0.32105 to 0.20541, saving model to D:/baseline_improvement-main/params_reassessment_june2026/ml_artifacts/v3\baseline_v3.keras
76/76 [==============================] - 4s 46ms/step - loss: 0.2745 - val_loss: 0.2054
Epoch 3/50
76/76 [==============================] - ETA: 0s - loss: 0.1973
Epoch 3: val_loss improved from 0.20541 to 0.15532, saving model to D:/baseline_improvement-main/params_reassessment_june2026/ml_artifacts/v3\baseline_v3.keras
76/76 [==============================] - 4s 52ms/step - loss: 0.1973 - val_loss: 0.1553
Epoch 4/50
76/76 [===============

In [38]:
model = LSTMTimeSeriesAutoencoder(sequence_length= X_train.shape[1], num_dimensions=X_train.shape[2], dim_weights = [0.80, 0.05, 0.05, 0.05, 0.025, 0.025])
# Load model weights 
model.load_weights('D:/baseline_improvement-main/params_reassessment_june2026/ml_artifacts/v3/baseline_v3.keras')

In [39]:
model.model.to_json()

'{"class_name": "Functional", "config": {"name": "model_3", "trainable": true, "layers": [{"module": "keras.layers", "class_name": "InputLayer", "config": {"batch_input_shape": [null, 10, 6], "dtype": "float32", "sparse": false, "ragged": false, "name": "input_4"}, "registered_name": null, "name": "input_4", "inbound_nodes": []}, {"module": "keras.layers", "class_name": "LSTM", "config": {"name": "lstm_30", "trainable": true, "dtype": "float32", "return_sequences": true, "return_state": false, "go_backwards": false, "stateful": false, "unroll": false, "time_major": false, "units": 128, "activation": "tanh", "recurrent_activation": "sigmoid", "use_bias": true, "kernel_initializer": {"module": "keras.initializers", "class_name": "GlorotUniform", "config": {"seed": null}, "registered_name": null}, "recurrent_initializer": {"module": "keras.initializers", "class_name": "Orthogonal", "config": {"gain": 1.0, "seed": null}, "registered_name": null}, "bias_initializer": {"module": "keras.initi

In [40]:
predicted_load_train = model.predict(X_train)

168/168 [==============================] - 6s 13ms/step


In [41]:
X_train[10:20,4,0]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [42]:
predicted_load_train[10:20, 4, 0]

array([-0.01418878, -0.0151948 , -0.01596738, -0.01652162, -0.01687946,
       -0.01706691, -0.01711084, -0.0167339 , -0.01589818, -0.01443253],
      dtype=float32)

In [43]:
def get_mean_and_std(x,y,t):
    error = x[:,t,0] - y[:, t, 0]
    mean_rmse = np.sqrt(np.mean(np.square(error)))
    std_rmse = np.std(error)
    return mean_rmse, std_rmse
    

In [44]:
def mark_anomaly(x, y, t, mean, std, relax=1):
    error = x[:,t,0] - y[:, t, 0]
    deviation = np.abs(error)

    threshold = mean + relax*std
    anomalies = deviation > threshold

    # print(mean_rmse, std_rmse, error, deviation)
    # print(anomalies)
    return anomalies

In [45]:
mean, std =  get_mean_and_std(X_train, predicted_load_train, 4)
print(mean, std)

0.06620750521802131 0.06605166587623582


In [46]:
anomaly = mark_anomaly(X_train, predicted_load_train, 4, mean, std)

In [ ]:
# Model conversion to ONNX: 
# !pip install tf2onnx onnxruntime --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 3.20.3 which is incompatible.


In [47]:
def display_prediction(df10_trainX, predicted_load_train, t, mean, std, relax):
    
    anomaly=mark_anomaly(df10_trainX, predicted_load_train, t, mean, std, relax)
    # print()
    # Create DataFrames for actual and predicted values
    # print(anomaly)
    df1 = pd.DataFrame({'x': range(len(df10_trainX)), 'y': df10_trainX[:, t, 0], 'color': 'Actual', 'anomaly': anomaly})
    df2 = pd.DataFrame({'x': range(len(predicted_load_train)), 'y': predicted_load_train[:, t, 0], 'color': 'Predicted', 'anomaly': False})
    # df3 = pd.DataFrame({'x': range(len(predicted_load_train)), 'y': predicted_load_train[:, t, 0], 'color': 'Anomaly', 'anomaly':anomaly})
    
    # Concatenate the DataFrames
    df = pd.concat([df1, df2])
    # print(df)
    # Plot both actual and predicted values
    fig = px.line(df, x='x', y='y', color='color')

    df_anomaly = df[df['anomaly']==True]
    print("Number of anomalies: ", len(df_anomaly))
    scatter_data = px.scatter(df_anomaly, x='x', y='y', color_discrete_sequence=['black']).data[0]
    scatter_data.update(marker=dict(size=5))
    fig.add_trace(scatter_data)

    fig.show()


In [48]:
display_prediction(X_train, predicted_load_train, 4, mean, std, 2)

Number of anomalies:  111


In [49]:
predicted_load_test = model.predict(X_test)

40/40 [==============================] - 1s 20ms/step


In [ ]:
# results on test data 

In [50]:
display_prediction(X_test, predicted_load_test, 4, mean, std, 2)

Number of anomalies:  12


In [51]:
type(X_train)

numpy.ndarray

In [ ]:
# # --- Save artifacts for export & inference ---
# import os
# import json
# import pickle

# os.makedirs("deployment_work/artifacts/latest", exist_ok=True)
# ART_DIR = "deployment_work/artifacts/latest"

# # 4) Save the scaler used in training
# with open(os.path.join(ART_DIR, "scaler.pkl"), "wb") as f:
#     pickle.dump(scaler2, f)

# # 6) Save calibration windows (raw inputs) and corresponding predictions for reproducible calibration
# np.save(os.path.join(ART_DIR, "training_set.npy"), df10_trainX)
# np.save(os.path.join(ART_DIR, "training_set_reconstructions.npy"), predicted_load_train) 

# # COmpute upper limit and lowr limit to save in metadata.josn 
# upper_limit = float(mean + 2 * std)
# lower_limit = float(mean - 2 * std)

# metadata = {
#     "anomaly_index": int(4),
#     "mean_error": mean,
#     "std_error": std,
#     "upper_limit": upper_limit,
#     "lower_limit": lower_limit,
# }
# with open(os.path.join(ART_DIR, "metadata.json"), "w") as f:
#     json.dump(metadata, f, indent=2)

# # 9) Save feature ordering & window config (very important)
# feature_config = {
#     "feature_schema": ['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1'],
#     "target_feature_index": 0,
#     "sequence_length": int(df10_trainX.shape[1]),
#     "num_features": int(df10_trainX.shape[2]), 
#     "dim_weights": [0.8, 0.05, 0.05, 0.05, 0.025], 
#     "num_calibration_samples": int(len(df10_trainX))
# }
# with open(os.path.join(ART_DIR, "feature_config.json"), "w") as f:
#     json.dump(feature_config, f, indent=2)

# print("Saved artifacts to", ART_DIR)

# np.save(os.path.join(ART_DIR, "test_dataset.npy"), df10_testX)

Saved artifacts to deployment_work/artifacts/latest


In [52]:
def duplicate_rows_with_probability(arr, probability=0.1, duplicate_count=50):
    duplicated_rows = []
    for row in arr:
        t = np.random.rand()
        if t < probability:
            for i in range(duplicate_count):
                duplicated_rows.append(row)
            # duplicated_rows.append(row)
        elif t >= probability and t < min(0.5, 2*probability):
            pass
        else:
            duplicated_rows.append(row)
        duplicated_rows.append(row)
            
    duplicated_arr = np.array(duplicated_rows)
    return duplicated_arr 

In [54]:
def graph_for_dilation(dfN10_train_data_scaled_2, prob, duplicate_count, mean, std, relax):
    dilated_test = duplicate_rows_with_probability(dfN10_train_data_scaled_2, prob, duplicate_count)
    # print(dilated_test.shape)
    # df10_trainX_3, _ = to_sequences(dilated_test, 10)
    # print(df10_trainX_3.shape)
    predicted_load_train_3 = model.predict(dilated_test)
    anomalies=mark_anomaly(dilated_test, predicted_load_train_3, 4, mean, std, relax)
    display_prediction(dilated_test, predicted_load_train_3, 4, mean, std, relax)

In [55]:
graph_for_dilation(X_test, 0.3, 2, mean, std, 2)

83/83 [==============================] - 1s 10ms/step
Number of anomalies:  25


In [ ]:
# impact with time dilation

In [56]:
graph_for_dilation(X_test, 0, 0, mean, std, 2)

79/79 [==============================] - 1s 11ms/step
Number of anomalies:  24


In [57]:
def get_noise(x, noise_level=0.1):
    size = len(x)
    noise = np.random.normal(loc=0, scale=noise_level, size=x.shape)
    return noise
    

In [58]:
noise = get_noise(X_test, 0.00)

In [59]:
predicted_load_test_noise = model.predict(X_test + noise)

40/40 [==============================] - 0s 11ms/step


In [60]:
display_prediction(X_test + noise, predicted_load_test_noise, 4,  mean, std, 2)

Number of anomalies:  12


In [61]:
# impact with synthetic noise

In [62]:
noise = get_noise(X_test, 0.1)
predicted_load_test_noise = model.predict(X_test + noise)
display_prediction(X_test + noise, predicted_load_test_noise, 4,  mean, std, 2)

40/40 [==============================] - 0s 11ms/step
Number of anomalies:  84


In [63]:
def get_patch_noise(x, random_patches=None, reduce_intensity=1):
    size = len(x)
    noise = np.zeros_like(x)
    # noise = np.ones(x.shape)/reduce_intensity
    if random_patches:
        for start, end in random_patches:
            patch_size = end - start
            print([*x.shape[-2:]] + [patch_size])
            # patch_noise = np.random.random(*x.shape[-2:], patch_size)
            patch_noise = np.random.random([*x.shape[-2:]] + [patch_size])/reduce_intensity
            # patch_noise = np.ones([*x.shape[-2:]] + [patch_size])/reduce_intensity
            # print(patch_noise)
            noise[start:end] = np.transpose(patch_noise, axes=(2, 0, 1)) 
    return noise


In [64]:
def get_ones_noise(x, random_patches=None, reduce_intensity=1):
    size = len(x)
    noise = np.zeros_like(x)
    noise = np.ones(x.shape)/reduce_intensity
    # if random_patches:
    #     for start, end in random_patches:
    #         patch_size = end - start
    #         print([*x.shape[-2:]] + [patch_size])
    #         # patch_noise = np.random.random(*x.shape[-2:], patch_size)
    #         patch_noise = np.random.random([*x.shape[-2:]] + [patch_size])/reduce_intensity
    #         # patch_noise = np.ones([*x.shape[-2:]] + [patch_size])/reduce_intensity
    #         # print(patch_noise)
    #         noise[start:end] = np.transpose(patch_noise, axes=(2, 0, 1))  
    
    return noise


In [65]:
# impact with inflated load points 

In [66]:
noise = get_ones_noise(X_test, [(50, 70), (300, 350)], 5)
# print(noise)
predicted_load_test_noise = model.predict(X_test + noise)
display_prediction(X_test + noise, predicted_load_test_noise, 4,  mean, std, 2)

40/40 [==============================] - 0s 11ms/step
Number of anomalies:  149


In [67]:
# impact with patch noise

In [68]:
noise = get_patch_noise(X_test, [(70, 100), (400, 450), (600, 670), (800, 900)], 2)
noise_data = X_test + noise
predicted_load_test_noise = model.predict(noise_data)
display_prediction(noise_data, predicted_load_test_noise, 4,  mean, std, 2)

[10, 6, 30]
[10, 6, 50]
[10, 6, 70]
[10, 6, 100]
40/40 [==============================] - 1s 12ms/step
Number of anomalies:  99


In [69]:
noise = get_patch_noise(X_test, [(70, 100), (400, 450), (600, 670), (800, 900)], 2)
# print(noise)
noise_data = X_test - noise
predicted_load_test_noise = model.predict(noise_data)
display_prediction(noise_data, predicted_load_test_noise, 4,  mean, std, 2)

[10, 6, 30]
[10, 6, 50]
[10, 6, 70]
[10, 6, 100]
40/40 [==============================] - 1s 16ms/step
Number of anomalies:  79


In [76]:
def get_data(machines, programs, dates):
    # machines= ['SL40309_015']
    # programs= ['O0005(5303-005-C)']
    # nsequences= ['N1']
    # dates= ['CR1_to_CR4', 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
    base_name = 'data'
    paths = []
    
    # data_SL40309_015_O0005(5303-005-C)_CR12_to_CR16
    
    for machine in machines:
        for program in programs:
                for date in dates:
                    path = f'{base_name}_{machine}_{program}_{date}.csv'
                    paths.append(path)
    print(paths)
    
    dfs = []
    
    for path in paths:
        data = pd.read_csv(f'../data/{path}')
        dfs.append(data)
        # print(len(data))
        
    data = pd.concat(dfs, ignore_index=True)
    
    data = data.pivot(index=['custom_id', 'timestamp', 'status', 'cr'], columns=['name', 'workstationcomponent'], values='value')
    data = data.reset_index()
    data.columns = [f'{col}_{comp}' if comp != '' else col for col, comp in data.columns]

    split_columns = data['status'].str.split('/', expand=True)
    data[['program_name', 'nsequence', 'execution']] = split_columns[[0,1,2]]
    
    data['load_Rotary_C5'] = data['load_Rotary_C5'].astype(float)
    data['position_Linear_Z'] = data['position_Linear_Z'].astype(float)
    data['position_Linear_X'] = data['position_Linear_X'].astype(float)
    data['spindlespeed_actual_Rotary_C5'] = data['spindlespeed_actual_Rotary_C5'].astype(float)
    data['pathfeedrate_Path_Path_1'] = data['pathfeedrate_Path_Path_1'].astype(float)
    
    data = data.ffill()
    return data

In [ ]:
# machines_diff = ['NL300005_007']
# programs_diff = ['O4073(3980-050-C)']
# dates_diff = ['CR1_to_CR4'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
# # data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5
# # data_NL300005_007_O4073(3980-050-C)_CR1_to_CR4

# data = get_data(machines_diff, programs_diff, dates_diff)

In [77]:
# raw_data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19

machines_diff = ['SL40309_015']
programs_diff = ['O0020(5304-020-F)']
dates_diff = ['N10_2024-03-19'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
# data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5
# data_NL300005_007_O4073(3980-050-C)_CR1_to_CR4

data = get_data(machines_diff, programs_diff, dates_diff)

['data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv']


FileNotFoundError: [Errno 2] No such file or directory: '../data/data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv'

In [158]:
len(data)

25024

In [159]:
def check_model_on_different_program(machines, programs, dates):
    data = get_data(machines_diff, programs_diff, dates_diff)
    data = data[(data['execution'] == 'ACTIVE') & 
            (data['spindlespeed_actual_Rotary_C5'] != 0) & 
            (data['pathfeedrate_Path_Path_1'] != 0) & 
            (data['load_Rotary_C5'] > 0) & 
            (data['pathfeedrate_Path_Path_1'] <= 1000)]
    
    # data_c = data[data['program_name'].isin(['O4125(5211-020-D)'])]
    # dfN10 = data_c[data_c['nsequence'].isin(['N1'])]
    data.dropna(subset=['load_Rotary_C5'],inplace=True)
    
    df_diff = data[['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]
    df_diff = df_diff.drop_duplicates(subset=['timestamp'], keep ='first')
    df_diff.reset_index(drop=True)

    df_diff['index'] = df_diff.groupby(['cr']).cumcount()
    
    fig = px.line(df_diff, x="index", y="load_Rotary_C5", color='cr')
    fig.update_layout(title_text="Load Vs Index")
    fig.show()

    df_diff['execution'] = LE.fit_transform(df_diff['execution'])
    df_diff.fillna(0, inplace=True)

    df_diff[['load_Rotary_C5']]= scaler1.fit_transform(df_diff[['load_Rotary_C5']])

    df_diff = df_diff[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]

    # scalar = MinMaxScaler()
    # data_scaled = scaler.fit(df_diff)
    # df_diff_scaled = data_scaled.transform(df_diff)
    df_diff_scaled = data_scaled.transform(df_diff)
    print(f"Max Before {df_diff.max()}")
    print(f"Max After {df_diff_scaled.max()}")
    # df_diff_scaled[-3]
    df10_trainX_diff, df10_trainY_diff = to_sequences(df_diff_scaled, 10)
    predicted_diff = model.predict(df10_trainX_diff)
    # print(predicted_diff[0:10])
    display_prediction(df10_trainX_diff, predicted_diff, 4, mean, std, 2)
    


In [160]:
machines_diff = ['SL40309_015']
programs_diff = ['O0020(5304-020-F)']
dates_diff = ['N10_2024-03-19'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
check_model_on_different_program(machines_diff, programs_diff, dates_diff)

['data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv']


Max Before load_Rotary_C5                     1.0000
position_Linear_X                 24.8753
position_Linear_Z                 17.7050
spindlespeed_actual_Rotary_C5     57.0000
pathfeedrate_Path_Path_1         259.5800
execution                          0.0000
dtype: float64
Max After 1.2516257125810348
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Number of anomalies:  0


In [161]:
machines_diff = ['NL250007_013']
programs_diff = ['O4125(5211-020-D)']
# data_SL40309_015_O0005(5303-005-C)_CR1_to_CR4
# data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5
dates_diff = ['N1_cr_1_to_5']
check_model_on_different_program(machines_diff, programs_diff, dates_diff)

['data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5.csv']


Max Before load_Rotary_C5                     1.00000
position_Linear_X                453.49922
position_Linear_Z                320.95440
spindlespeed_actual_Rotary_C5     73.00000
pathfeedrate_Path_Path_1         629.00000
execution                          0.00000
dtype: float64
Max After 22.741812274694034
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Number of anomalies:  3412


In [168]:
machines_diff = ['SL40309_015']
programs_diff = ['O0020(5304-020-F)']
dates_diff = ['N10_2024-03-19'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']

# machines_diff = ['SL40309_015']
# programs_diff = ['O0005(5303-005-C)']
# # data_SL40309_015_O0005(5303-005-C)_CR1_to_CR4
# dates_diff = ['CR1_to_CR4', 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']

data = get_data(machines_diff, programs_diff, dates_diff)
data = data[(data['execution'] == 'ACTIVE') & 
        (data['spindlespeed_actual_Rotary_C5'] != 0) & 
        (data['pathfeedrate_Path_Path_1'] != 0) & 
        (data['load_Rotary_C5'] > 0) & 
        (data['pathfeedrate_Path_Path_1'] <= 1000)]

# data_c = data[data['program_name'].isin(['O4125(5211-020-D)'])]
# dfN10 = data_c[data_c['nsequence'].isin(['N1'])]
data.dropna(subset=['load_Rotary_C5'],inplace=True)

df_diff = data[['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]
df_diff = df_diff.drop_duplicates(subset=['timestamp'], keep ='first')
df_diff.reset_index(drop=True)

df_diff['index'] = df_diff.groupby(['cr']).cumcount()

df_diff['execution'] = LE.fit_transform(df_diff['execution'])
df_diff.fillna(0, inplace=True)
# scaler1 = MinMaxScaler()

# df_diff[['load_Rotary_C5']]= scaler2.fit_transform(df_diff[['load_Rotary_C5']])
df_diff = df_diff[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]

# scalar = MinMaxScaler()
# data_scaled = scaler.fit(df_diff)
df_diff_scaled = scaler2.transform(df_diff)


['data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv']


In [169]:
df_diff_scaled.max()

np.float64(1.2516257125810348)

In [170]:
df_diff_scaled

array([[ 2.74509804e-01,  1.02633730e+00,  1.07283146e+00,
         7.65550239e-02,  1.06574739e+00,  0.00000000e+00],
       [ 3.92156863e-02,  7.60693103e-01,  8.31153613e-01,
         7.65550239e-02,  1.06574739e+00,  0.00000000e+00],
       [ 3.92156863e-02,  7.60693103e-01,  5.46059041e-01,
         7.89473684e-02,  1.06566433e+00,  0.00000000e+00],
       ...,
       [ 3.92156863e-02,  7.22984593e-01,  2.56378325e-01,
         5.98086124e-02, -8.30668273e-05,  0.00000000e+00],
       [ 1.96078431e-02,  7.22508285e-01,  3.01931879e-01,
         5.02392344e-02,  7.80329775e-01,  0.00000000e+00],
       [ 1.96078431e-02,  7.64684058e-01,  5.81236329e-01,
         4.78468900e-02,  9.83428168e-01,  0.00000000e+00]],
      shape=(3482, 6))

In [171]:
df_diff

,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution
18,15.0,20.3819,17.7050,45.0,256.83,0
23,3.0,15.0836,12.4458,45.0,256.83,0
29,3.0,15.0836,6.2418,46.0,256.81,0
88,3.0,11.4611,2.0000,46.0,236.20,0
93,3.0,11.4611,-0.0620,46.0,236.20,0
...,...,...,...,...,...,...
24773,3.0,14.3431,-0.0620,38.0,0.21,0
24775,3.0,14.3315,-0.0620,38.0,0.21,0
24776,3.0,14.3315,-0.0620,38.0,0.21,0
24987,2.0,14.3220,0.9293,34.0,188.11,0


In [172]:
df_diff.max()

load_Rotary_C5                    43.0000
position_Linear_X                 24.8753
position_Linear_Z                 17.7050
spindlespeed_actual_Rotary_C5     57.0000
pathfeedrate_Path_Path_1         259.5800
execution                          0.0000
dtype: float64

In [173]:
df10_trainX_diff, df10_trainY_diff = to_sequences(df_diff_scaled, 10)
predicted_diff = model.predict(df10_trainX_diff)
display_prediction(df10_trainX_diff, predicted_diff, 4, mean, std, 2)

109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Number of anomalies:  13
